# 04 — Estudio comparativo de reducción 512→64

Objetivo: comparar 5 generadores de los 64 features intermedios del pipeline del Frente 2 con la misma cabeza simbólica KAN[64,32,2], para decidir cuál da mejor fórmula simbólica utilizable.

Metodología (control perfecto): cada variante genera features 64-D z-score (stats de train) sobre `f2base` (GAP 512-D); luego la cabeza (fit 10 + máscara 30, poda por fidelidad, fix simbólico completa y compacta, Config C/D corregida, smoke-test) es idéntica en todas — vía el módulo reutilizable `src/explainability/f2_symb.py`.

Variantes:

| tag | técnica | supervisada | no-lineal | fuente features |
|---|---|---|---|---|
| `lin` | Bottleneck Linear 512→64 | sí | no | N19 (`f2bottleneck`) |
| `kan0` | KAN0[512,64] no simbólica | sí | sí | N20 (`f2kred`) |
| `pca` | PCA 64 comps | no | no | este notebook |
| `ae` | Autoencoder denso 512→128→64 | no | sí | este notebook |
| `mlp` | MLP bottleneck supervisado 512→128→64 | sí | sí | este notebook |

Salidas: features `f2pca_*`, `f2ae_*`, `f2mlp_*` y JSON consolidado `frente2/reduccion_comparativa.json`.


## 1. Setup


In [ ]:
import os, sys, re, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve

for p in (os.environ.get("TESIS_CODIGOS", os.path.abspath("..")), os.environ.get("TESIS_REP", os.path.abspath(os.path.join("..", "artifacts")))):
    if p not in sys.path:
        sys.path.insert(0, p)
from src.explainability.f2_symb import SymbHead, softmax2, youden_threshold

CODIGOS = os.environ.get("TESIS_CODIGOS", os.path.abspath(".."))
REP = os.environ.get("TESIS_REP", os.path.abspath(os.path.join("..", "artifacts")))
EMB = os.path.join(REP, "output", "embeddings")
OUT = os.path.join(REP, "output", "frente2")
FIGS = os.path.join(CODIGOS, "reports", "figures")
for d in (EMB, OUT, FIGS):
    os.makedirs(d, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
print("device:", DEVICE, "| pytorch:", torch.__version__)

## 2. Datos base (f2base 512-D) y saneamiento


In [2]:
BASE = "f2base"
feats512, labs512 = {}, {}
for split in ("train", "val", "test"):
    feats512[split] = np.load(os.path.join(EMB, f"{BASE}_{split}_feats.npy")).astype(np.float32)
    labs512[split] = np.load(os.path.join(EMB, f"{BASE}_{split}_labels.npy")).astype(int)
    assert feats512[split].shape[0] == labs512[split].shape[0], f"{split} no alineado"
    assert np.isfinite(feats512[split]).all(), f"{split} NaN/inf"
    print(f"{split:5s} {feats512[split].shape} clases={np.bincount(labs512[split]).tolist()}")

mu512 = feats512["train"].mean(0, keepdims=True)
sd512 = feats512["train"].std(0, keepdims=True) + 1e-8
Z512 = {s: (feats512[s] - mu512) / sd512 for s in feats512}

cnt = np.bincount(labs512["train"]).astype(np.float64)
clw = 1.0 / cnt
clw = clw / clw.sum()
class_weights = torch.tensor(clw, dtype=torch.float32, device=DEVICE)
print("class_weights:", class_weights.tolist())

train (4278, 512) clases=[2439, 1839]
val   (1134, 512) clases=[631, 503]
test  (1012, 512) clases=[594, 418]
class_weights: [0.4298737645149231, 0.5701262354850769]


In [3]:
def run_variant(tag, feats64, labs, out_row=True):
    """Corre la cabeza symólica KAN[64,32,2] sobre features 64-D z-score.

    feats64: dict split -> (N,64) float32 numpy YA z-score
    labs:    dict split -> (N,) int numpy
    """
    x64 = {s: torch.from_numpy(feats64[s]).to(DEVICE) for s in feats64}
    y64 = {s: torch.from_numpy(labs[s]).to(DEVICE) for s in labs}

    head = SymbHead(in_dim=64, hidden=32, grid=3, k=3, seed=SEED, device=DEVICE)
    print(f"\n=== VARIANTE {tag} ===")
    auc_val, auc_test = head.fit_and_sparsify(
        x64["train"], y64["train"], x64["val"], y64["val"],
        x64["test"], y64["test"], fit_ep=10, sparse_ep=30,
        class_weights=class_weights)
    print(f"  Config A: AUC val={auc_val:.4f} test={auc_test:.4f}")

    D_sweet, kept_dims, tabla = head.prune_sweep(
        x64["val"], x64["test"], y64["val"], y64["test"], auc_val)
    print("  Barrido D':", {r['D']: (r['r'], r['dAUC_val']) for r in tabla})
    print(f"  D'= {D_sweet}, kept_dims={kept_dims[:8]}...")

    # --- completa ---
    kp_full = head.build_pruned(kept_dims)
    with torch.no_grad():
        kp_full(x64["val"])
    kp_full.symbolic_enabled = True
    n1 = head.fix_layer1(kp_full)
    n0 = head.fix_layer0(kp_full, x_val=x64["val"])
    kp_full.symbolic_enabled = False
    fben_full, fmal_full = head.make_formula(kp_full)

    with torch.no_grad():
        Lv_ref = head.kan(x64["val"]).cpu().numpy()
    Cv_full, Ct_full, rC_full = head.config_C(
        kp_full, x64["val"], x64["test"], y64["val"], y64["test"], Lv_ref)

    kept_arr = np.array(kept_dims)
    Xv_kept = feats64["val"][:, kept_arr]
    Xt_kept = feats64["test"][:, kept_arr]
    yv64 = labs["val"].astype(np.float64)
    yb64 = labs["test"].astype(np.float64)
    D_full = head.report_D(fben_full, fmal_full, Xv_kept, Xt_kept, yv64, yb64)
    ok, diff = head.smoke_test(D_full["auc_D"], Ct_full)

    # --- compacta ---
    kp_cmp = head.build_pruned(kept_dims)
    with torch.no_grad():
        kp_cmp(x64["val"])
    kp_cmp.symbolic_enabled = True
    n1c = head.fix_layer1(kp_cmp)
    n0c = head.fix_layer0(kp_cmp, top_k=12, x_val=x64["val"])
    kp_cmp.symbolic_enabled = False
    fben_cmp, fmal_cmp = head.make_formula(kp_cmp)
    Cv_cmp, Ct_cmp, rC_cmp = head.config_C(
        kp_cmp, x64["val"], x64["test"], y64["val"], y64["test"], Lv_ref)
    D_cmp = head.report_D(fben_cmp, fmal_cmp, Xv_kept, Xt_kept, yv64, yb64)

    print(f"  [C completa] test={Ct_full:.4f} r={rC_full:.4f} | "
          f"[C compacta] test={Ct_cmp:.4f} r={rC_cmp:.4f}")
    print(f"  [D completa] {D_full} | [D compacta] {D_cmp}")
    print(f"  formulas completa={len(fmal_full)}B/{len(fben_full)}B "
          f"compacta={len(fmal_cmp)}B/{len(fben_cmp)}B")

    # scores de test de la formula para graficar ROC
    st_full = head.eval_formula_vec(fmal_full, Xt_kept) - head.eval_formula_vec(fben_full, Xt_kept)
    st_cmp = head.eval_formula_vec(fmal_cmp, Xt_kept) - head.eval_formula_vec(fben_cmp, Xt_kept)

    return {
        "tag": tag,
        "auc_val": round(auc_val, 4), "auc_test": round(auc_test, 4),
        "D_sweet": D_sweet,
        "prune_table": tabla,
        "C_full_val": round(Cv_full, 4), "C_full_test": round(Ct_full, 4),
        "r_C_full": round(rC_full, 4),
        "C_cmp_val": round(Cv_cmp, 4), "C_cmp_test": round(Ct_cmp, 4),
        "r_C_cmp": round(rC_cmp, 4),
        "D_full": D_full, "D_cmp": D_cmp,
        "smoke_ok": ok, "smoke_diff": round(diff, 4),
        "len_ben_full": len(fben_full), "len_mal_full": len(fmal_full),
        "len_ben_cmp": len(fben_cmp), "len_mal_cmp": len(fmal_cmp),
        "n_var_ben_full": head.leg(fben_full)["n_vars"],
        "n_var_mal_full": head.leg(fmal_full)["n_vars"],
        "n_var_ben_cmp": head.leg(fben_cmp)["n_vars"],
        "n_var_mal_cmp": head.leg(fmal_cmp)["n_vars"],
        "kb_ben_full": head.leg(fben_full)["kb"], "kb_mal_full": head.leg(fmal_full)["kb"],
        "kb_ben_cmp": head.leg(fben_cmp)["kb"], "kb_mal_cmp": head.leg(fmal_cmp)["kb"],
        "formula_ben_full": fben_full[:3000], "formula_mal_full": fmal_full[:3000],
        "formula_ben_cmp": fben_cmp[:3000], "formula_mal_cmp": fmal_cmp[:3000],
        "kept_dims": kept_dims,
        "st_full_test": st_full.tolist(), "st_cmp_test": st_cmp.tolist(),
    }

## 3. Variantes lin (N19) y kan0 (N20) — features ya guardadas

Se cargan features 64-D ya z-scoreadas de notebooks previos.


In [4]:
# Tags a comparar. Para lin y kan0 los feature 64-D ya existen (z-scoreados).
# pca, ae, mlp se generan a continuacion.
variants = ["lin", "kan0"]
feats_cache = {}
labs_cache = {}

for tag, prefix in (("lin", "f2bottleneck"), ("kan0", "f2kred")):
    feats, labs = {}, {}
    for s in ("train", "val", "test"):
        feats[s] = np.load(os.path.join(EMB, f"{prefix}_{s}_feats.npy")).astype(np.float32)
        labs[s] = np.load(os.path.join(EMB, f"{prefix}_{s}_labels.npy")).astype(int)
    feats_cache[tag] = feats
    labs_cache[tag] = labs
    print(f"{tag}: {feats['train'].shape} (de {prefix})")

lin: (4278, 64) (de f2bottleneck)
kan0: (4278, 64) (de f2kred)


## 4. Variante PCA (no supervisado)

Genera features 64-D con `PCA(n=64)` ajustado SOLO sobre train. Guarda `f2pca_*`.


In [5]:
from sklearn.decomposition import PCA

t0 = time.perf_counter()
pca = PCA(n_components=64, random_state=SEED).fit(Z512["train"])  # fit SOLO train
t_pca = time.perf_counter() - t0
exp_var = pca.explained_variance_ratio_

# Aplicar a val/test y z-score con stats de train
feats_pca = {s: pca.transform(Z512[s]) for s in ("train", "val", "test")}
mu_p = feats_pca["train"].mean(0, keepdims=True)
sd_p = feats_pca["train"].std(0, keepdims=True) + 1e-8
feats_pca = {s: ((feats_pca[s] - mu_p) / sd_p).astype(np.float32) for s in feats_pca}

for s in ("train", "val", "test"):
    np.save(os.path.join(EMB, f"f2pca_{s}_feats.npy"), feats_pca[s])
    np.save(os.path.join(EMB, f"f2pca_{s}_labels.npy"), labs512[s].astype(int))
variants.append("pca")
feats_cache["pca"] = feats_pca
labs_cache["pca"] = labs512

print(f"PCA: {t_pca:.3f}s | varianza 64 comps: {100*exp_var.sum():.1f}% | top16 {100*exp_var[:16].sum():.1f}%")

PCA: 0.050s | varianza 64 comps: 87.4% | top16 73.3%


## 5. Variante Autoencoder denso (no supervisado, MSE)

`512 -> 128 -> 64 (bottleneck) -> 128 -> 512`. Entrena sobre train; extrae el bottleneck 64-D.


In [6]:
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(512, 128), nn.ReLU(),
            nn.Linear(128, 64),          # bottleneck 64-D
        )
        self.dec = nn.Sequential(
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, 512),
        )
    def forward(self, x):
        z = self.enc(x)
        return self.dec(z), z

ae = Autoencoder().to(DEVICE)
x_tr_t = torch.from_numpy(Z512["train"]).to(DEVICE)
opt_ae = torch.optim.Adam(ae.parameters(), lr=1e-3, weight_decay=1e-4)
loss_ae = nn.MSELoss()

t0 = time.perf_counter()
print("AE fit (20 epocas)")
for ep in range(1, 21):
    ae.train()
    perm = torch.randperm(len(x_tr_t), device=DEVICE)
    tot, nb = 0.0, 0
    for i in range(0, len(x_tr_t), 512):
        idx = perm[i:i+512]
        opt_ae.zero_grad()
        rec, _ = ae(x_tr_t[idx])
        loss = loss_ae(rec, x_tr_t[idx])
        loss.backward(); opt_ae.step()
        tot += loss.item() * len(idx); nb += len(idx)
    if ep % 5 == 0:
        print(f"  ep {ep:02d}: mse {tot/nb:.6f}")
t_ae = time.perf_counter() - t0
print(f"AE fit: {t_ae:.1f}s")

AE fit (20 epocas)
  ep 05: mse 0.429372
  ep 10: mse 0.303844
  ep 15: mse 0.255087
  ep 20: mse 0.227477
AE fit: 0.8s


In [7]:
# Extraer bottleneck 64-D del AE y z-score con stats de train
ae.eval()
with torch.no_grad():
    z_ae = {s: ae(torch.from_numpy(Z512[s]).to(DEVICE))[1].cpu().numpy()
            for s in ("train", "val", "test")}
mu_a = z_ae["train"].mean(0, keepdims=True)
sd_a = z_ae["train"].std(0, keepdims=True) + 1e-8
feats_ae = {s: ((z_ae[s] - mu_a) / sd_a).astype(np.float32) for s in z_ae}

for s in ("train", "val", "test"):
    np.save(os.path.join(EMB, f"f2ae_{s}_feats.npy"), feats_ae[s])
    np.save(os.path.join(EMB, f"f2ae_{s}_labels.npy"), labs512[s].astype(int))
variants.append("ae")
feats_cache["ae"] = feats_ae
labs_cache["ae"] = labs512
print("AE features 64-D guardadas.")

AE features 64-D guardadas.


## 6. Variante MLP supervisado (CE con clase)

`512 -> 128 -> 64 (bottleneck) -> 2`. Entrena con CrossEntropy y pesos de clase; la activación
del bottleneck 64-D es la representación de la que se extraerá la fórmula.


In [8]:
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

class MLPBottleneck(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(512, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),   # bottleneck 64-D
            nn.Linear(64, 2),
        )
    def forward(self, x):
        return self.net(x)
    def encode(self, x):
        return self.net[:3](x)  # hasta la activación del bottleneck

mlp = MLPBottleneck().to(DEVICE)
y_tr_t = torch.from_numpy(labs512["train"]).to(DEVICE)
opt_mlp = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
crit_mlp = nn.CrossEntropyLoss(weight=class_weights)

t0 = time.perf_counter()
print("MLP fit (20 epocas)")
for ep in range(1, 21):
    mlp.train()
    perm = torch.randperm(len(x_tr_t), device=DEVICE)
    tot, nb = 0.0, 0
    for i in range(0, len(x_tr_t), 512):
        idx = perm[i:i+512]
        opt_mlp.zero_grad()
        out = mlp(x_tr_t[idx])
        loss = crit_mlp(out, y_tr_t[idx])
        loss.backward(); opt_mlp.step()
        tot += loss.item() * len(idx); nb += len(idx)
    # AUC val
    if ep % 5 == 0:
        mlp.eval()
        with torch.no_grad():
            pv = torch.softmax(mlp(torch.from_numpy(Z512["val"]).to(DEVICE)), 1)[:, 1].cpu().numpy()
        auc_v = roc_auc_score(labs512["val"], pv)
        print(f"  ep {ep:02d}: loss {tot/nb:.4f} | AUC val {auc_v:.4f}")
t_mlp = time.perf_counter() - t0
print(f"MLP fit: {t_mlp:.1f}s")

MLP fit (20 epocas)
  ep 05: loss 0.1437 | AUC val 0.7738
  ep 10: loss 0.1039 | AUC val 0.7735
  ep 15: loss 0.0625 | AUC val 0.7684
  ep 20: loss 0.0248 | AUC val 0.7660
MLP fit: 0.4s


In [9]:
# Extraer bottleneck 64-D del MLP y z-score con stats de train
mlp.eval()
with torch.no_grad():
    z_mlp = {s: mlp.encode(torch.from_numpy(Z512[s]).to(DEVICE)).cpu().numpy()
             for s in ("train", "val", "test")}
mu_m = z_mlp["train"].mean(0, keepdims=True)
sd_m = z_mlp["train"].std(0, keepdims=True) + 1e-8
feats_mlp = {s: ((z_mlp[s] - mu_m) / sd_m).astype(np.float32) for s in z_mlp}

for s in ("train", "val", "test"):
    np.save(os.path.join(EMB, f"f2mlp_{s}_feats.npy"), feats_mlp[s])
    np.save(os.path.join(EMB, f"f2mlp_{s}_labels.npy"), labs512[s].astype(int))
variants.append("mlp")
feats_cache["mlp"] = feats_mlp
labs_cache["mlp"] = labs512
print("MLP features 64-D guardadas.")

MLP features 64-D guardadas.


## 7. Ejecutar la cabeza simbólica en cada variante (harness idéntico)


In [10]:
results = {}
for tag in variants:
    results[tag] = run_variant(tag, feats_cache[tag], labs_cache[tag])
    print()

checkpoint directory created: ./model
saving model version 0.0

=== VARIANTE lin ===
  [fit] ep 05: AUC test=0.7285 vivas=2112
  [fit] ep 10: AUC test=0.7271 vivas=2112
  [sparsify] ep 05: AUC test=0.7260 vivas=1794
  [sparsify] ep 10: AUC test=0.7256 vivas=1477
  [sparsify] ep 15: AUC test=0.7252 vivas=1161
  [sparsify] ep 20: AUC test=0.7252 vivas=844
  [sparsify] ep 25: AUC test=0.7253 vivas=528
  [sparsify] ep 30: AUC test=0.7216 vivas=210
  Config A: AUC val=0.7696 test=0.7216
  Barrido D': {48: (0.9995, -0.0022), 32: (0.9944, -0.0009), 24: (0.9908, -0.0005), 16: (0.9613, 0.0046)}
  D'= 48, kept_dims=[1, 2, 3, 4, 5, 6, 9, 10]...


<venv>\lib\site-packages\sympy\core\sympify.py:475: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:821.)
  return sympify(float(a))


  |AUC_D(0.7251) - AUC_C(0.7251)| = 0.0000 -> OK
  [C completa] test=0.7251 r=0.9877 | [C compacta] test=0.7257 r=0.9938
  [D completa] {'auc_D': 0.7251, 'th_youden_val': 0.375, 'J_youden_val': 0.3838, 'recall_D_youden': 0.6364, 'spec_D_youden': 0.6936} | [D compacta] {'auc_D': 0.7136, 'th_youden_val': -0.1302, 'J_youden_val': 0.3846, 'recall_D_youden': 0.866, 'spec_D_youden': 0.3653}
  formulas completa=3929B/1626B compacta=734B/18B

checkpoint directory created: ./model
saving model version 0.0

=== VARIANTE kan0 ===
  [fit] ep 05: AUC test=0.7248 vivas=2112
  [fit] ep 10: AUC test=0.7231 vivas=2112
  [sparsify] ep 05: AUC test=0.7205 vivas=1794
  [sparsify] ep 10: AUC test=0.7188 vivas=1477
  [sparsify] ep 15: AUC test=0.7177 vivas=1161
  [sparsify] ep 20: AUC test=0.7171 vivas=844
  [sparsify] ep 25: AUC test=0.7163 vivas=528
  [sparsify] ep 30: AUC test=0.7151 vivas=210
  Config A: AUC val=0.7620 test=0.7151
  Barrido D': {48: (0.9978, -0.0005), 32: (0.9874, -0.0006), 24: (0.9722,

<venv>\lib\site-packages\numpy\_core\_methods.py:135: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
<venv>\lib\site-packages\numpy\_core\_methods.py:135: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
<venv>\lib\site-packages\numpy\_core\_methods.py:135: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
<venv>\lib\site-packages\numpy\_core\_methods.py:135: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
<venv>\lib\site-packages\numpy\_core\_methods.py:135: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)


  |AUC_D(0.7134) - AUC_C(0.7133)| = 0.0001 -> OK


<venv>\lib\site-packages\numpy\_core\_methods.py:135: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
<venv>\lib\site-packages\numpy\_core\_methods.py:135: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
<venv>\lib\site-packages\numpy\_core\_methods.py:135: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
<venv>\lib\site-packages\numpy\_core\_methods.py:135: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
<venv>\lib\site-packages\numpy\_core\_methods.py:135: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)


  [C completa] test=0.7133 r=0.9156 | [C compacta] test=0.7096 r=0.9818
  [D completa] {'auc_D': 0.7134, 'th_youden_val': -0.2536, 'J_youden_val': 0.398, 'recall_D_youden': 0.7129, 'spec_D_youden': 0.5842} | [D compacta] {'auc_D': 0.7135, 'th_youden_val': 0.1868, 'J_youden_val': 0.3905, 'recall_D_youden': 0.7297, 'spec_D_youden': 0.5438}
  formulas completa=1742B/498B compacta=197B/17B



## 8. Tabla resumen comparativa


In [11]:
import pandas as pd

rows = []
for tag, r in results.items():
    rows.append({
        "variante": tag,
        "AUC_full": r["auc_test"],
        "D_sweet": r["D_sweet"],
        "C_full_test": r["C_full_test"],
        "D_full_test": r["D_full"]["auc_D"],
        "D_cmp_test": r["D_cmp"]["auc_D"],
        "len_mal_full": r["len_mal_full"],
        "nvar_mal_full": r["n_var_mal_full"],
        "kb_mal_full": r["kb_mal_full"],
        "len_mal_cmp": r["len_mal_cmp"],
        "nvar_mal_cmp": r["n_var_mal_cmp"],
        "kb_mal_cmp": r["kb_mal_cmp"],
        "recall_D_full": r["D_full"]["recall_D_youden"],
        "spec_D_full": r["D_full"]["spec_D_youden"],
        "smoke_ok": r["smoke_ok"],
    })
df_res = pd.DataFrame(rows)
print(df_res.to_string(index=False))

variante  AUC_full  D_sweet  C_full_test  D_full_test  D_cmp_test  len_mal_full  nvar_mal_full  kb_mal_full  len_mal_cmp  nvar_mal_cmp  kb_mal_cmp  recall_D_full  spec_D_full  smoke_ok
     lin    0.7216       48       0.7251       0.7251      0.7136          3929             42          3.8          734             9         0.7         0.6364       0.6936      True
    kan0    0.7151       48       0.7212       0.7213      0.7234          2223             46          2.2          341            11         0.3         0.6746       0.5976      True
     pca    0.7244       48       0.7278       0.7342      0.7153           772             15          0.8          150             1         0.1         0.6292       0.6852      True
      ae    0.7253       48       0.7218       0.7210      0.7028          1709             29          1.7          203             4         0.2         0.8110       0.4529      True
     mlp    0.7094       48       0.7133       0.7134      0.7135          

## 9. Guardar JSON consolidado + ROCs


In [12]:
out_path = os.path.join(OUT, "reduccion_comparativa.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("Guardado en", out_path)

# ROCs de la Config D (compacta y completa) por variante sobre test
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

yb64 = labs512["test"].astype(np.float64)
plt.figure(figsize=(8, 7))
for tag in variants:
    r = results[tag]
    st = np.array(r["st_cmp_test"])
    fpr, tpr, _ = roc_curve(yb64, st)
    auc_v = roc_auc_score(yb64, st)
    plt.plot(fpr, tpr, label=f"{tag}: compacta AUC={auc_v:.3f}")
plt.plot([0, 1], [0, 1], '--', color="gray", alpha=0.6)
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.legend(loc="lower right")
plt.title("N21 — Curvas ROC de la fórmula compacta por variante (test)")
plt.savefig(os.path.join(FIGS, "21_roc_reduccion_variantes.png"), dpi=150, bbox_inches="tight")
plt.close()
print("ROC guardado en reports/figures/21_roc_reduccion_variantes.png")

Guardado en <repo>/artifacts\output\frente2\reduccion_comparativa.json
ROC guardado en reports/figures/21_roc_reduccion_variantes.png
